# Chapter 08 — Give It a Leash

**Secure Agentic AI Developer Workbook**

> OWASP: LLM10 (Unbounded Consumption) · LLM06 (Excessive Agency) · ASI09 (Human–Agent Trust — confirmation gates)

In this lab you will:
1. **Runaway loop** — watch an agent without a step limit iterate past 30 steps; apply the hard ceiling fix
2. **Token overrun** — simulate a high-token run with no budget; apply mid-run enforcement
3. **Tool call storm** — trigger a burst of `send_email` calls; watch the rate limiter block the excess
4. **Irreversible action without confirmation** — watch a delete-all execute instantly; apply HITL + blast radius cap

No API key required — all exercises use mock LLM responses and simulated tool calls.


In [ ]:
%pip install -q colorama

In [ ]:
import time
from collections import defaultdict
from dataclasses import dataclass
from enum import Enum
from threading import Lock
from typing import Callable
from colorama import Fore, Style, init
init(autoreset=True)

def ok(msg):    print(Fore.GREEN  + "✅ " + msg)
def bad(msg):   print(Fore.RED    + "💥 " + msg)
def info(msg):  print(Fore.CYAN   + "ℹ  " + msg)
def warn(msg):  print(Fore.YELLOW + "⚠  " + msg)
def sep(t):     print(Fore.YELLOW + "=" * 60 + "\n" + t + "\n" + "=" * 60)
def sec(t):     print(Fore.GREEN  + "=" * 60 + "\n" + t + "\n" + "=" * 60)

# ── Constants ────────────────────────────────────────────────────────────────
MAX_STEPS    = 15
WARN_STEPS   = 10
TOKEN_BUDGET = 20_000    # lower than chapter for faster lab demo
COST_PER_1K  = 0.01
MAX_COST_USD = 0.20

# ── Mock LLM helpers ─────────────────────────────────────────────────────────
class FakeCall:
    def __init__(self, name, args): self.name = name; self.args = args

class FakeResponse:
    def __init__(self, tool_calls, content, tokens):
        self.tool_calls = tool_calls
        self.content    = content
        class Usage: pass
        u = Usage()
        u.total_tokens = tokens
        self.usage = u

def mock_execute_tool(name, args):
    return f"[result of {name}({args})]"

ok("Setup complete.")

---
## Exercise 1 — Runaway Agent Loop

Without a step limit, an agent loops as long as the model keeps requesting tool calls. Watch it run past 30 steps on an impossible task, then apply the hard ceiling.


In [ ]:
# Mock LLM that loops 40 times before completing
def mock_llm_looping(messages, _counter=[0]):
    _counter[0] += 1
    n = _counter[0]
    if n >= 40:
        return FakeResponse(None, f"Done after {n} steps.", 500)
    return FakeResponse([FakeCall("search_web", {"q": f"topic {n}"})], None, 1200)

# ❌ VULNERABLE: no step limit
def run_agent_vuln(user_request):
    messages = [{"role": "user", "content": user_request}]
    mock_llm_looping.__defaults__ = None  # reset
    mock_llm_looping.__code__ = mock_llm_looping.__code__  # no-op reset trick
    counter = [0]
    def _llm(msgs):
        counter[0] += 1
        n = counter[0]
        if n >= 40:
            return FakeResponse(None, f"Done after {n} steps.", 500)
        return FakeResponse([FakeCall("search_web", {"q": f"topic {n}"})], None, 1200)

    while True:
        response = _llm(messages)
        if response.tool_calls:
            for call in response.tool_calls:
                messages.append({"role": "tool", "content": mock_execute_tool(call.name, call.args)})
        else:
            return {"steps": counter[0], "result": response.content}

sep("EXERCISE 1a — Agent without step limit")
print()
result = run_agent_vuln("Research everything about quantum computing")
bad(f"Agent completed after {result['steps']} steps — no ceiling enforced")
info("In production, an impossible task would loop until infrastructure timeout (minutes)")

In [ ]:
# ✅ Best Practice: hard step ceiling
def run_agent_secure(user_request, max_steps=MAX_STEPS):
    messages = [{"role": "user", "content": user_request}]
    steps    = 0
    counter  = [0]

    def _llm(msgs):
        counter[0] += 1
        n = counter[0]
        if n >= 40:   # model would complete at 40 — but we stop it at MAX_STEPS
            return FakeResponse(None, f"Done after {n} steps.", 500)
        return FakeResponse([FakeCall("search_web", {"q": f"topic {n}"})], None, 1200)

    while steps < max_steps:
        steps += 1
        if steps == WARN_STEPS:
            warn(f"  [WARN] Approaching step limit: {steps}/{max_steps}")

        response = _llm(messages)
        if not response.tool_calls:
            return {"status": "completed", "steps": steps, "result": response.content}

        for call in response.tool_calls:
            messages.append({"role": "tool", "content": mock_execute_tool(call.name, call.args)})

    return {"status": "incomplete", "steps": steps,
            "result": "Step limit reached. Please refine the request."}

sec("EXERCISE 1b — Agent with hard step ceiling (MAX_STEPS=15)")
print()
result = run_agent_secure("Research everything about quantum computing")
print()
if result["status"] == "incomplete":
    ok(f"Agent halted cleanly after {result['steps']} steps — status: incomplete")
    ok(f"Result message: {result['result']}")
else:
    ok(f"Completed in {result['steps']} steps")
info("Runaway loop impossible — ceiling enforced in code, not by infrastructure timeout")

---
## Exercise 2 — Token Budget Overrun

Each step adds tokens. Without a budget check, costs compound silently. Apply mid-run token enforcement.


In [ ]:
def mock_llm_expensive(messages, _c=[0]):
    """Each call costs 3,000 tokens — expensive model on long context."""
    _c[0] += 1
    if _c[0] >= 12:
        return FakeResponse(None, "Done.", 800)
    return FakeResponse([FakeCall("search_web", {"q": f"step {_c[0]}"})], None, 3000)

# ❌ VULNERABLE: no budget enforcement
def run_no_budget(user_request):
    messages, tokens, steps = [{"role": "user", "content": user_request}], 0, 0
    _c = [0]
    def _llm(m):
        _c[0] += 1
        if _c[0] >= 12: return FakeResponse(None, "Done.", 800)
        return FakeResponse([FakeCall("search_web", {"q": f"step {_c[0]}"})], None, 3000)

    while steps < 20:
        steps += 1
        r = _llm(messages)
        tokens += r.usage.total_tokens
        if not r.tool_calls:
            return {"status": "completed", "tokens": tokens,
                    "cost_usd": round(tokens/1000*COST_PER_1K, 4)}
        for c in r.tool_calls:
            messages.append({"role": "tool", "content": mock_execute_tool(c.name, c.args)})
    return {"status": "incomplete", "tokens": tokens}

# ✅ Best Practice: budget checked before each call
def run_with_budget(user_request):
    messages, tokens, steps = [{"role": "user", "content": user_request}], 0, 0
    _c = [0]
    def _llm(m):
        _c[0] += 1
        if _c[0] >= 12: return FakeResponse(None, "Done.", 800)
        return FakeResponse([FakeCall("search_web", {"q": f"step {_c[0]}"})], None, 3000)

    while steps < MAX_STEPS:
        steps += 1
        est = sum(len(m.get("content",""))//4 for m in messages)
        if tokens + est > TOKEN_BUDGET:
            return {"status": "budget_exceeded", "tokens": tokens,
                    "cost_usd": round(tokens/1000*COST_PER_1K, 4),
                    "result": "Token budget reached."}
        r = _llm(messages)
        tokens += r.usage.total_tokens
        cost = tokens/1000*COST_PER_1K
        if cost > MAX_COST_USD:
            return {"status": "cost_limit_reached", "tokens": tokens,
                    "cost_usd": round(cost, 4), "result": "Cost ceiling hit."}
        if not r.tool_calls:
            return {"status": "completed", "steps": steps, "tokens": tokens,
                    "cost_usd": round(cost, 4), "result": r.content}
        for c in r.tool_calls:
            messages.append({"role": "tool", "content": mock_execute_tool(c.name, c.args)})
    return {"status": "incomplete", "tokens": tokens}

sep("EXERCISE 2a — No budget enforcement")
print()
r = run_no_budget("Write a comprehensive 50-page report")
bad(f"Run completed: tokens={r['tokens']:,}  cost=${r.get('cost_usd',0):.4f}")
if r.get('tokens', 0) > TOKEN_BUDGET:
    bad(f"Exceeded TOKEN_BUDGET ({TOKEN_BUDGET:,}) by {r['tokens']-TOKEN_BUDGET:,} tokens")

print()
sec("EXERCISE 2b — Mid-run budget enforcement")
print()
r2 = run_with_budget("Write a comprehensive 50-page report")
status_colour = Fore.GREEN if r2['status'] in ('completed','budget_exceeded','cost_limit_reached') else Fore.RED
print(f"  Status : {status_colour}{r2['status']}{Style.RESET_ALL}")
print(f"  Tokens : {r2['tokens']:,} (budget={TOKEN_BUDGET:,})")
print(f"  Cost   : ${r2.get('cost_usd',0):.4f} (ceiling=${MAX_COST_USD})")
if r2['tokens'] <= TOKEN_BUDGET or r2['status'] in ('budget_exceeded','cost_limit_reached'):
    ok("Agent stopped before exceeding limits — mid-run enforcement works")

---
## Exercise 3 — Tool Call Rate Limit

An agent that can call `send_email` without limits will eventually be used for spam — or overwhelm an external API. Apply the per-tool rate limiter.


In [ ]:
TOOL_RATE_LIMITS = {
    "send_email":  {"calls_per_minute": 2, "max_per_run": 3},
    "search_web":  {"calls_per_minute": 10, "max_per_run": 20},
    "default":     {"calls_per_minute": 20, "max_per_run": 40},
}

class ToolRateLimiter:
    def __init__(self):
        self._times  = defaultdict(list)
        self._counts = defaultdict(int)
        self._lock   = Lock()

    def check_and_record(self, tool):
        lim = TOOL_RATE_LIMITS.get(tool, TOOL_RATE_LIMITS["default"])
        with self._lock:
            if self._counts[tool] >= lim["max_per_run"]:
                raise RuntimeError(f"run_limit_exceeded: {tool!r} ({lim['max_per_run']}/run)")
            now    = time.monotonic()
            cutoff = now - 60.0
            self._times[tool] = [t for t in self._times[tool] if t > cutoff]
            if len(self._times[tool]) >= lim["calls_per_minute"]:
                raise RuntimeError(f"rate_limit_exceeded: {tool!r} ({lim['calls_per_minute']}/min)")
            self._times[tool].append(now)
            self._counts[tool] += 1

EMAIL_LOG = []

sep("EXERCISE 3a — Unrestricted send_email calls")
print()
calls_attempted = 8
info(f"Agent attempts {calls_attempted} send_email calls with no rate limiter:")
for i in range(calls_attempted):
    EMAIL_LOG.append(f"email_{i+1}")
    bad(f"  send_email call {i+1}: sent")
bad(f"Total emails sent: {len(EMAIL_LOG)} — no limit enforced")

print()
sec("EXERCISE 3b — Same calls through rate limiter (max_per_run=3)")
print()
limiter   = ToolRateLimiter()
sent, blocked = 0, 0
for i in range(calls_attempted):
    try:
        limiter.check_and_record("send_email")
        sent += 1
        ok(f"  send_email call {i+1}: allowed ({sent}/3)")
    except RuntimeError as e:
        blocked += 1
        warn(f"  send_email call {i+1}: BLOCKED — {e}")

print()
ok(f"Sent: {sent}  Blocked: {blocked}  (cap=3/run)")
info("Rate limiter stopped the burst — external email API never received calls 4–8")

---
## Exercise 4 — Irreversible Action Without Confirmation

An agent executes a delete operation immediately. Apply the HITL confirmation gate and blast radius cap.


In [ ]:
class ActionRisk(Enum):
    SAFE         = "safe"
    REVERSIBLE   = "reversible"
    IRREVERSIBLE = "irreversible"

TOOL_RISK = {
    "search_web":     ActionRisk.SAFE,
    "query_database": ActionRisk.SAFE,
    "delete_records": ActionRisk.IRREVERSIBLE,
    "send_email":     ActionRisk.IRREVERSIBLE,
}

MAX_DELETE_ROWS   = 100
MAX_EMAILS_PER_RUN = 3

@dataclass
class PendingAction:
    tool_name: str
    args: dict
    risk: ActionRisk
    preview: str

DELETED, EMAILS_SENT = [], []

def mock_delete(table, limit=99999, **kw):
    rows = [f"row_{i}" for i in range(min(limit, 40000))]
    DELETED.extend(rows)
    return f"Deleted {len(rows)} rows from {table}"

def mock_send(to, **kw):
    EMAILS_SENT.append(to)
    return f"Email sent to {to}"

CLEANUP_CALLS = [
    FakeCall("delete_records", {"table": "orders", "limit": 40000}),
    FakeCall("send_email",     {"to": "all-staff@company.com", "subject": "Done"}),
]

# ❌ VULNERABLE
sep("EXERCISE 4a — Irreversible actions fire immediately")
print()
DELETED.clear(); EMAILS_SENT.clear()
for call in CLEANUP_CALLS:
    if call.name == "delete_records": result = mock_delete(**call.args)
    elif call.name == "send_email":    result = mock_send(**call.args)
    else:                              result = mock_execute_tool(call.name, call.args)
    bad(f"Executed {call.name}: {result}")
bad(f"Total rows deleted: {len(DELETED):,}  Emails sent: {len(EMAILS_SENT)}")
bad("No confirmation requested. No blast radius check. No undo.")

print()
# ✅ Best Practice
sec("EXERCISE 4b — Same actions through HITL gate + blast radius cap")
print()
DELETED.clear(); EMAILS_SENT.clear()
limiter2 = ToolRateLimiter()

def execute_with_hitl(tool_calls, confirm_fn):
    pending      = [PendingAction(c.name, c.args,
                                  TOOL_RISK.get(c.name, ActionRisk.IRREVERSIBLE),
                                  f"{c.name}({c.args})") for c in tool_calls]
    irreversible = [a for a in pending if a.risk == ActionRisk.IRREVERSIBLE]

    # Blast radius cap
    for a in irreversible:
        if "delete" in a.tool_name:
            n = int(a.args.get("limit", 0))
            if n > MAX_DELETE_ROWS:
                raise ValueError(f"blast_radius_exceeded: {n} rows > cap of {MAX_DELETE_ROWS}")
    emails = sum(1 for a in irreversible if "email" in a.tool_name)
    if emails > MAX_EMAILS_PER_RUN:
        raise ValueError(f"blast_radius_exceeded: {emails} emails > cap of {MAX_EMAILS_PER_RUN}")

    if irreversible:
        approved = confirm_fn(irreversible)
        if not approved:
            raise PermissionError("human_rejected_irreversible_actions")

    results = []
    for a in pending:
        limiter2.check_and_record(a.tool_name)
        if a.tool_name == "delete_records": results.append(mock_delete(**a.args))
        elif a.tool_name == "send_email":    results.append(mock_send(**a.args))
        else:                               results.append(mock_execute_tool(a.tool_name, a.args))
    return results

# Scenario A: blast radius cap fires before confirmation
info("Scenario A: agent proposes deleting 40,000 rows")
try:
    execute_with_hitl(CLEANUP_CALLS, confirm_fn=lambda actions: True)
    bad("Should have been blocked")
except ValueError as e:
    ok(f"Blast radius cap fired BEFORE confirmation: {e}")
    ok(f"Rows deleted: {len(DELETED)}  (cap prevented confirmation from even appearing)")

print()
# Scenario B: within cap, user rejects
info("Scenario B: agent proposes deleting 50 rows (within cap) — user rejects")
SAFE_CALLS = [
    FakeCall("delete_records", {"table": "orders", "limit": 50}),
]
try:
    execute_with_hitl(SAFE_CALLS, confirm_fn=lambda actions: (
        warn(f"  Human sees: {[a.preview for a in actions]}") or False
    ))
    bad("Should have been rejected")
except PermissionError as e:
    ok(f"Human rejected: {e}")
    ok(f"Rows deleted: {len(DELETED)} — no action taken after rejection")

---
## Summary

| Vulnerability | Root cause | Fix |
|---|---|---|
| Runaway loop | No step ceiling — loops until infra timeout | Hard `MAX_STEPS` ceiling; structured `incomplete` result on breach |
| Token overrun | Usage tracked but never enforced mid-run | Check budget before each LLM call; separate USD cost ceiling |
| Tool call storm | No per-tool rate limit | Sliding-window limiter: calls/min + calls/run per tool |
| Irreversible action without review | Delete/send/pay execute immediately | Blast radius cap + HITL confirmation gate before execution |

**The key principle:** The model optimises for task completion. It will not stop because it has been running too long, spent too much, or is about to do something irreversible. Those constraints are your responsibility as the developer — enforce them in code, not in the system prompt.

---

**Next:** [Chapter 09 — Vet Your Plugins](../chapters/09-vet-your-plugins.md) *(coming soon)*
